In [ ]:
import pandas as pd

df = pd.read_csv("sentiment_reviews.csv")
print(df.head())

                                              Review Sentiment
0  A very, very, very slow-moving, aimless movie ...  Negative
1  Not sure who was more lost - the flat characte...  Negative
2  Attempting artiness with black & white and cle...  Negative
3         Very little music or anything to speak of.  Negative
4  The best scene in the movie was when Gerardo i...  Positive


In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate reviews: {duplicate_count}")

Number of duplicate reviews: 3


In [ ]:
sentiment_counts = df['Sentiment'].value_counts()
print(sentiment_counts)

Sentiment
Negative    500
Positive    500
Name: count, dtype: int64


In [ ]:
import torch
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
from torch.nn.functional import softmax
from sklearn.metrics import classification_report
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

# Device config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model and tokenizer
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, output_attentions=True)
model.to(device)
model.eval()

# Read and clean dataset
df = pd.read_csv("sentiment_reviews.csv")
df['Sentiment'] = df['Sentiment'].str.strip().str.lower()  # Normalize capitalization and whitespace

positive_reviews = df[df["Sentiment"] == "positive"]["Review"].tolist()
negative_reviews = df[df["Sentiment"] == "negative"]["Review"].tolist()

reviews = positive_reviews + negative_reviews
labels = [1] * len(positive_reviews) + [0] * len(negative_reviews)

label_map = {"positive": 1, "negative": 0}
stop_words = set(stopwords.words("english"))

# Function to get attention weights
def get_attention_weights(attentions):
    # Use attentions from the last layer, average over all heads
    last_layer_attn = attentions[-1]  # shape: (batch_size, num_heads, seq_len, seq_len)
    avg_attn = last_layer_attn.mean(dim=1)  # average across heads
    cls_attn = avg_attn[:, 0, :]  # attention from CLS token
    return cls_attn[0].cpu().numpy()

# Process individual review and get token attention values
def process_review(review):
    inputs = tokenizer(review, return_tensors="pt", truncation=True, max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        attentions = outputs.attentions

    attention_weights = get_attention_weights(attentions)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])[1:-1]  # Exclude [CLS] and [SEP]

    if not tokens or not attention_weights.any():
        print("⚠️ Warning: No valid tokens or attention weights for review:", review[:50])
        return {}

    word_values = {}
    for token, weight in zip(tokens, attention_weights[1:-1]):
        if token not in stop_words:
            word_values[token] = word_values.get(token, 0) + float(weight)
    return word_values

# Store results
results = []
attention_data = []

for idx, review in enumerate(reviews):
    actual_label = labels[idx]

    # Predict label
    inputs = tokenizer(review, return_tensors="pt", truncation=True, max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        prediction = torch.argmax(softmax(logits, dim=1), dim=1).item()

    # Store prediction result
    results.append({
        "Review": review,
        "Actual Label": actual_label,
        "Predicted Label": prediction
    })

    # Store attention weights
    word_weights = process_review(review)
    attention_data.append(word_weights)

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv("results.csv", index=False)

attention_df = pd.DataFrame(attention_data).fillna(0)
attention_df.to_csv("bert_attention_weights.csv", index=False)

# Report
y_true = [row["Actual Label"] for row in results]
y_pred = [row["Predicted Label"] for row in results]
print(classification_report(y_true, y_pred))


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSelfAttention.forward`.
  return forward_call(*args, **kwargs)


              precision    recall  f1-score   support

           0       0.49      0.96      0.65       500
           1       0.21      0.01      0.02       500

    accuracy                           0.49      1000
   macro avg       0.35      0.49      0.34      1000
weighted avg       0.35      0.49      0.34      1000



In [ ]:
#l2 norm method
import torch
import numpy as np
import pandas as pd
from transformers import DistilBertTokenizer, DistilBertModel
import nltk
from nltk.corpus import stopwords

# Download stopwords if needed
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Load pre-trained BERT model and tokenizer
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
model = DistilBertModel.from_pretrained(model_name, output_attentions=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Load dataset
try:
    df = pd.read_csv("sentiment_reviews.csv")
except FileNotFoundError:
    print("Error: The file 'sentiment_reviews.csv' was not found.")
    exit()
except pd.errors.EmptyDataError:
    print("Error: The file is empty.")
    exit()
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    exit()

# Function to extract term weights using BERT embeddings
def get_term_weights(review):
    inputs = tokenizer(review, return_tensors="pt", truncation=True, max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        hidden_states = outputs.last_hidden_state  # Extract last hidden states

    term_weights = torch.norm(hidden_states, dim=-1).squeeze().cpu().numpy()  # Compute L2 norm for term weights
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])[1:-1]  # Remove CLS & SEP

    word_values = {tok: weight for tok, weight in zip(tokens, term_weights) if tok not in stop_words}
    return word_values

# Process all reviews and aggregate term weights
overall_term_weights = {}
for review in df["review"].tolist():
    word_weights = get_term_weights(review)
    for word, weight in word_weights.items():
        overall_term_weights[word] = overall_term_weights.get(word, 0) + weight

# Merge data into a CSV-compatible format
data = [{"Term": word, "Term Weight": weight} for word, weight in overall_term_weights.items()]

# Save to CSV
df_output = pd.DataFrame(data)
df_output.to_csv("bert_term_weights2.csv", index=False)

print("CSV file 'bert_term_weights.csv' created successfully!")



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

CSV file 'bert_term_weights.csv' created successfully!


In [ ]:
#hidden state
import torch
import numpy as np
import pandas as pd
from transformers import DistilBertTokenizer, DistilBertModel
import nltk
from nltk.corpus import stopwords

# Download stopwords if needed
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Load fine-tuned sentiment model
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
model = DistilBertModel.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Load dataset
try:
    df = pd.read_csv("sentiment_reviews.csv")
except FileNotFoundError:
    print("Error: The file 'sentiment_reviews.csv' was not found.")
    exit()
except pd.errors.EmptyDataError:
    print("Error: The file is empty.")
    exit()
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    exit()

# Function to process a review and extract term weights
def process_review(review):
    inputs = tokenizer(review, return_tensors="pt", truncation=True, max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        hidden_states = outputs.last_hidden_state  # Shape: (batch_size, sequence_length, hidden_size)

    # Average the hidden states across the sequence length
    term_weights = hidden_states.mean(dim=1).squeeze().cpu().numpy()  # Shape: (hidden_size,)

    # Get tokens and their corresponding term weights
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])[1:-1]  # Remove CLS & SEP
    token_weights = {tok: term_weights[i] for i, tok in enumerate(tokens) if tok not in stop_words}

    return token_weights

# Process positive and negative reviews
positive_reviews = df[df["sentiment"] == "positive"]["review"].tolist()
negative_reviews = df[df["sentiment"] == "negative"]["review"].tolist()

overall_word_values = {}
positive_word_values = {}
negative_word_values = {}

# Process all reviews and aggregate term weights
for review in positive_reviews:
    word_weights = process_review(review)
    for word, weight in word_weights.items():
        positive_word_values[word] = positive_word_values.get(word, 0) + weight
        overall_word_values[word] = overall_word_values.get(word, 0) + weight

for review in negative_reviews:
    word_weights = process_review(review)
    for word, weight in word_weights.items():
        negative_word_values[word] = negative_word_values.get(word, 0) + weight
        overall_word_values[word] = overall_word_values.get(word, 0) + weight

# Merge data into a CSV-compatible format
all_words = set(positive_word_values.keys()).union(set(negative_word_values.keys()))
data = []
for word in all_words:
    data.append({
        "Term": word,
        "Term Weight (Positive Review)": positive_word_values.get(word, 0),
        "Term Weight (Negative Review)": negative_word_values.get(word, 0),
        "Term Weight (Overall)": overall_word_values.get(word, 0)
    })

# Save to CSV
df_output = pd.DataFrame(data)
df_output.to_csv("bert_term_weightshidd.csv", index=False)

print("CSV file 'bert_term_weights.csv' created successfully!")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


CSV file 'bert_term_weights.csv' created successfully!


In [ ]:
#gradient
import torch
import pandas as pd
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

# Load fine-tuned sentiment model
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Load dataset
df = pd.read_csv("sentiment_reviews.csv")

# Function to compute word importance using gradients
def get_grad_importance(review):
    inputs = tokenizer(review, return_tensors="pt", truncation=True, max_length=512, padding=True)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    model.zero_grad()  # Reset gradients

    # Get embeddings and enable gradient tracking
    embeddings = model.distilbert.embeddings.word_embeddings(inputs["input_ids"])
    embeddings.retain_grad()

    # Forward pass
    outputs = model(inputs_embeds=embeddings, attention_mask=inputs["attention_mask"])
    loss = outputs.logits[:, 1].sum()  # Sentiment class 1 (positive)
    loss.backward()  # Compute gradients

    # Get gradients for word embeddings
    gradients = embeddings.grad.abs().sum(dim=-1).squeeze().cpu().numpy()  # Shape: (seq_len,)

    # Get tokens & their corresponding importance scores
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze().tolist())
    token_importance = {tok: gradients[i] for i, tok in enumerate(tokens) if tok not in ["[CLS]", "[SEP]", "[PAD]"]}

    return token_importance

# Compute word importance for all reviews
df["Token_Importance"] = df["review"].apply(get_grad_importance)

# Save results
df.to_csv("gradient_based_importance.csv", index=False)

print("Gradient-based term importance extracted successfully!")


Gradient-based term importance extracted successfully!


In [ ]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load BERT model and tokenizer
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

# Function to get the [CLS] embedding of a text
def get_cls_embedding(text):
    tokens = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        output = model(**tokens)
    return output.last_hidden_state[:, 0, :].squeeze().numpy()  # Extract [CLS] embedding

# Function to compute term weight using cosine similarity
def compute_term_weight(term, text_data):
    term_embedding = get_cls_embedding(term)
    doc_embedding = get_cls_embedding(text_data)
    similarity = cosine_similarity(term_embedding.reshape(1, -1), doc_embedding.reshape(1, -1))
    return similarity[0][0]  # Return cosine similarity score

# Load dataset
df = pd.read_csv("sentiment_reviews.csv")

# Separate positive and negative reviews
positive_reviews = " ".join(df[df["sentiment"] == "positive"]["review"])
negative_reviews = " ".join(df[df["sentiment"] == "negative"]["review"])
overall_document = " ".join(df["review"])  # All reviews combined

# Define the term for analysis
term = "quality"  # Change this to any term of interest

# Compute term weights
term_weight_pos = compute_term_weight(term, positive_reviews)
term_weight_neg = compute_term_weight(term, negative_reviews)
term_weight_overall = compute_term_weight(term, overall_document)

# Save results to a new CSV file
results = pd.DataFrame({
    "Term": [term],
    "Term Weight (Positive Reviews)": [term_weight_pos],
    "Term Weight (Negative Reviews)": [term_weight_neg],
    "Term Weight (Overall Document)": [term_weight_overall]
})

# Save to CSV
results.to_csv("term_weights.csv", index=False)

print("Term weights saved to 'term_weights.csv'")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Term weights saved to 'term_weights.csv'


In [ ]:
#cls for single term trial
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load BERT model and tokenizer
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

# Function to get [CLS] embedding of a text
def get_cls_embedding(text):
    tokens = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        output = model(**tokens)
    return output.last_hidden_state[:, 0, :].squeeze().numpy()  # Extract [CLS] embedding

# Function to compute term weight for a single review
def compute_term_weight(term, review):
    term_embedding = get_cls_embedding(term)
    review_embedding = get_cls_embedding(review)
    similarity = cosine_similarity(term_embedding.reshape(1, -1), review_embedding.reshape(1, -1))
    return similarity[0][0]  # Return cosine similarity score

# Load dataset (assuming "reviews.csv" contains "review" and "sentiment" columns)
df = pd.read_csv("sentiment_reviews.csv")

# Define the term for analysis
term = "quality"  # Change this to the word you want to analyze

# Lists to store term weights for each category
positive_weights = []
negative_weights = []
overall_weights = []

# Process each review
for _, row in df.iterrows():
    review_text = row["review"]
    term_weight = compute_term_weight(term, review_text)

    overall_weights.append(term_weight)  # Store for overall document

    # Categorize term weights
    if row["sentiment"] == "positive":
        positive_weights.append(term_weight)
    elif row["sentiment"] == "negative":
        negative_weights.append(term_weight)

# Compute average term weights
term_weight_pos = np.mean(positive_weights) if positive_weights else 0
term_weight_neg = np.mean(negative_weights) if negative_weights else 0
term_weight_overall = np.mean(overall_weights) if overall_weights else 0

# Save results to a new CSV file
results = pd.DataFrame({
    "Term": [term],
    "Term Weight (Positive Reviews)": [term_weight_pos],
    "Term Weight (Negative Reviews)": [term_weight_neg],
    "Term Weight (Overall Document)": [term_weight_overall]
})

# Save to CSV
results.to_csv("term_weightsclss.csv", index=False)

print("Term weights saved to 'term_weights.csv'")

Term weights saved to 'term_weights.csv'


In [ ]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re

# Load BERT model and tokenizer
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

# Function to get [CLS] embedding of a text
def get_cls_embedding(text):
    tokens = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        output = model(**tokens)
    return output.last_hidden_state[:, 0, :].squeeze().numpy()  # Extract [CLS] embedding

# Function to compute term weight for a single review
def compute_term_weight(term, review):
    term_embedding = get_cls_embedding(term)
    review_embedding = get_cls_embedding(review)
    similarity = cosine_similarity(term_embedding.reshape(1, -1), review_embedding.reshape(1, -1))
    return similarity[0][0]  # Return cosine similarity score

# Load dataset
df = pd.read_csv("sentiment_reviews.csv")

# Extract unique words from all reviews
all_text = " ".join(df["Review"]).lower()  # Combine all reviews into one string
words = set(re.findall(r"\b\w+\b", all_text))  # Extract unique words

# Dictionary to store results
results = {"Term": [], "Term Weight (Positive Reviews)": [], "Term Weight (Negative Reviews)": [], "Term Weight (Overall Document)": []}

# Process each word
for term in words:
    positive_weights = []
    negative_weights = []
    overall_weights = []

    # Process each review
    for _, row in df.iterrows():
        review_text = row["Review"]
        term_weight = compute_term_weight(term, review_text)

        overall_weights.append(term_weight)  # Store for overall document

        if row["Sentiment"] == "positive":
            positive_weights.append(term_weight)
        elif row["Sentiment"] == "negative":
            negative_weights.append(term_weight)

    # Compute average term weights
    term_weight_pos = np.mean(positive_weights) if positive_weights else 0
    term_weight_neg = np.mean(negative_weights) if negative_weights else 0
    term_weight_overall = np.mean(overall_weights) if overall_weights else 0

    # Store results
    results["Term"].append(term)
    results["Term Weight (Positive Reviews)"].append(term_weight_pos)
    results["Term Weight (Negative Reviews)"].append(term_weight_neg)
    results["Term Weight (Overall Document)"].append(term_weight_overall)

# Convert to DataFrame and save to CSV
df_results = pd.DataFrame(results)
df_results.to_csv("term_weights_all.csv", index=False)

print("Term weights saved to 'term_weights_all.csv'")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import nltk
from nltk.corpus import stopwords

# Download stopwords if needed
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Load fine-tuned sentiment model
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, output_attentions=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Load dataset
try:
    df = pd.read_csv("sentiment_reviews.csv")
except FileNotFoundError:
    print("Error: The file 'sentiment_reviews.csv' was not found.")
    exit()
except pd.errors.EmptyDataError:
    print("Error: The file is empty.")
    exit()
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    exit()

# Function to extract attention weights
def get_attention_weights(attentions):
    last_layers = attentions[-2:]  # Take last 2 layers for better accuracy
    attn_scores = torch.mean(torch.stack(last_layers), dim=[0, 1, 2])  # Mean over layers & heads
    term_weights = attn_scores[0, 1:-1].cpu().numpy()  # Remove CLS & SEP
    term_weights /= term_weights.sum()  # Normalize
    return term_weights

# Function to process a review and extract word-level attention weights
def process_review(review):
    inputs = tokenizer(review, return_tensors="pt", truncation=True, max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        attentions = outputs.attentions

    attention_weights = get_attention_weights(attentions)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])[1:-1]  # Remove CLS & SEP

    word_values = {tok: attn for tok, attn in zip(tokens, attention_weights) if tok not in stop_words}
    return word_values

# Process positive and negative reviews
positive_reviews = df[df["sentiment"] == "positive"]["review"].tolist()
negative_reviews = df[df["sentiment"] == "negative"]["review"].tolist()

overall_word_values = {}
positive_word_values = {}
negative_word_values = {}

# Store predicted sentiments
predicted_sentiments = []

# Process all reviews and aggregate attention weights
for index, row in df.iterrows():
    review = row["review"]
    word_weights = process_review(review)

    # Compute log-ratio sum for the review
    log_weighted_sum = sum(
        np.log((positive_word_values.get(word, 0) + 0.001) / (negative_word_values.get(word, 0) + 0.001))
        for word in word_weights.keys()
    )
    predicted_sentiment = "positive" if log_weighted_sum > 0 else "negative"
    predicted_sentiments.append(predicted_sentiment)

    for word, weight in word_weights.items():
        if row["sentiment"] == "positive":
            positive_word_values[word] = positive_word_values.get(word, 0) + weight
        else:
            negative_word_values[word] = negative_word_values.get(word, 0) + weight
        overall_word_values[word] = overall_word_values.get(word, 0) + weight

# Add predicted sentiment column to DataFrame
df["Predicted Sentiment"] = predicted_sentiments

# Prepare data for output with log-ratio calculations
all_words = set(positive_word_values.keys()).union(set(negative_word_values.keys()))
data = []
for word in all_words:
    positive_weight = positive_word_values.get(word, 0)
    negative_weight = negative_word_values.get(word, 0)
    overall_weight = overall_word_values.get(word, 0)
    log_ratio = np.log((positive_weight + 0.001) / (negative_weight + 0.001))

    data.append({
        "Term": word,
        "Attention Weight (Positive Review)": positive_weight,
        "Attention Weight (Negative Review)": negative_weight,
        "Attention Weight (Overall)": overall_weight,
        "Log Ratio": log_ratio
    })

# Save predicted sentiments and word-level attention weights to CSV
df.to_csv("bert_predicted_sentiments.csv", index=False)
df_output = pd.DataFrame(data)
df_output.to_csv("bert_attention_weights.csv", index=False)

print("CSV files 'bert_predicted_sentiments.csv' and 'bert_attention_weights.csv' created successfully!")


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

DistilBertSdpaAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


CSV files 'bert_predicted_sentiments.csv' and 'bert_attention_weights.csv' created successfully!


In [ ]:
#roberta
import torch
import numpy as np
import pandas as pd
from transformers import RobertaTokenizer, RobertaForSequenceClassification
import nltk
from nltk.corpus import stopwords
from sklearn.metrics import precision_score, recall_score, f1_score

# Download stopwords if needed
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Load RoBERTa sentiment model
model_name = "cardiffnlp/twitter-roberta-base-sentiment"  # Pretrained sentiment model
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained(model_name, output_attentions=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Load dataset
try:
    df = pd.read_csv("sentiment_reviews(imdb).csv")
except FileNotFoundError:
    print("Error: The file 'sentiment_reviews.csv' was not found.")
    exit()
except pd.errors.EmptyDataError:
    print("Error: The file is empty.")
    exit()
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    exit()

# Function to extract attention weights
def get_attention_weights(attentions):
    last_layers = attentions[-2:]  # Take last 2 layers for better accuracy
    attn_scores = torch.mean(torch.stack(last_layers), dim=[0, 1, 2])  # Mean over layers & heads
    term_weights = attn_scores[0, 1:-1].cpu().numpy()  # Remove CLS & SEP
    term_weights /= term_weights.sum()  # Normalize
    return term_weights

# Function to process a review and extract word-level attention weights
def process_review(review):
    inputs = tokenizer(review, return_tensors="pt", truncation=True, max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        attentions = outputs.attentions

    attention_weights = get_attention_weights(attentions)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])[1:-1]  # Remove CLS & SEP

    word_values = {tok: attn for tok, attn in zip(tokens, attention_weights) if tok not in stop_words}
    return word_values

# Process positive and negative reviews
positive_reviews = df[df["sentiment"] == "positive"]["review"].tolist()
negative_reviews = df[df["sentiment"] == "negative"]["review"].tolist()

overall_word_values = {}
positive_word_values = {}
negative_word_values = {}

# Store predicted and actual sentiments
predicted_sentiments = []
actual_sentiments = df["sentiment"].tolist()

# Process all reviews and aggregate attention weights
for index, row in df.iterrows():
    review = row["review"]
    word_weights = process_review(review)
    log_weighted_score = sum(
        np.log((positive_word_values.get(word, 0) + 0.001) / (negative_word_values.get(word, 0) + 0.001))
        for word in word_weights.keys()
    )
    predicted_sentiment = "positive" if log_weighted_score > 0 else "negative"
    predicted_sentiments.append(predicted_sentiment)

    for word, weight in word_weights.items():
        if row["sentiment"] == "positive":
            positive_word_values[word] = positive_word_values.get(word, 0) + weight
        else:
            negative_word_values[word] = negative_word_values.get(word, 0) + weight
        overall_word_values[word] = overall_word_values.get(word, 0) + weight

# Prepare data for output with an additional column for the formula
all_words = set(positive_word_values.keys()).union(set(negative_word_values.keys()))
data = []
for word in all_words:
    positive_weight = positive_word_values.get(word, 0)
    negative_weight = negative_word_values.get(word, 0)
    overall_weight = overall_word_values.get(word, 0)

    # Calculate the new column value using the provided formula
    new_column_value = np.log((positive_weight + 0.001) / (negative_weight + 0.001))

    data.append({
        "Term": word,
        "Attention Weight (Positive Review)": positive_weight,
        "Attention Weight (Negative Review)": negative_weight,
        "Attention Weight (Overall)": overall_weight,
        "Log Ratio": new_column_value  # New column added
    })

# Add predicted sentiment column to DataFrame
df["Predicted Sentiment"] = predicted_sentiments

# Calculate accuracy
df["Correct Prediction"] = df.apply(lambda row: 1 if row["Predicted Sentiment"] == row["sentiment"] else 0, axis=1)
accuracy = (df["Correct Prediction"].sum() / len(df)) * 100

# Calculate Precision, Recall, F1 Score
y_true = [1 if s == "positive" else 0 for s in actual_sentiments]
y_pred = [1 if s == "positive" else 0 for s in predicted_sentiments]

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

# Save predicted sentiments and accuracy results to CSV
df.to_csv("roberta_predicted_sentiments.csv", index=False)

print(f"Model accuracy: {accuracy:.2f}%")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print("CSV file 'roberta_predicted_sentiments.csv' updated successfully!")


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

An unexpected error occurred: 'utf-8' codec can't decode byte 0xc2 in position 1726: invalid continuation byte


NameError: name 'df' is not defined

In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import nltk
from nltk.corpus import stopwords
from sklearn.metrics import precision_score, recall_score, f1_score

# Download stopwords if needed
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Load fine-tuned sentiment model
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, output_attentions=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Load dataset
try:
    df = pd.read_csv("amazon_sentiment_reviews.csv")
except FileNotFoundError:
    print("Error: The file 'sentiment_reviews.csv' was not found.")
    exit()
except pd.errors.EmptyDataError:
    print("Error: The file is empty.")
    exit()
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    exit()

# Function to extract attention weights
def get_attention_weights(attentions):
    last_layers = attentions[-2:]  # Take last 2 layers for better accuracy
    attn_scores = torch.mean(torch.stack(last_layers), dim=[0, 1, 2])  # Mean over layers & heads
    term_weights = attn_scores[0, 1:-1].cpu().numpy()  # Remove CLS & SEP
    term_weights /= term_weights.sum()  # Normalize
    return term_weights

# Function to process a review and extract word-level attention weights
def process_review(review):
    inputs = tokenizer(review, return_tensors="pt", truncation=True, max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        attentions = outputs.attentions

    attention_weights = get_attention_weights(attentions)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])[1:-1]  # Remove CLS & SEP

    word_values = {tok: attn for tok, attn in zip(tokens, attention_weights) if tok not in stop_words}
    return word_values

# Process positive and negative reviews
positive_reviews = df[df["sentiment"] == "positive"]["review"].tolist()
negative_reviews = df[df["sentiment"] == "negative"]["review"].tolist()

overall_word_values = {}
positive_word_values = {}
negative_word_values = {}

# Store predicted and actual sentiments
predicted_sentiments = []
actual_sentiments = df["sentiment"].tolist()

# Process all reviews and aggregate attention weights
for index, row in df.iterrows():
    review = row["review"]
    word_weights = process_review(review)
    log_weighted_score = sum(
        np.log((positive_word_values.get(word, 0) + 0.001) / (negative_word_values.get(word, 0) + 0.001))
        for word in word_weights.keys()
    )
    predicted_sentiment = "positive" if log_weighted_score > 0 else "negative"
    predicted_sentiments.append(predicted_sentiment)

    for word, weight in word_weights.items():
        if row["sentiment"] == "positive":
            positive_word_values[word] = positive_word_values.get(word, 0) + weight
        else:
            negative_word_values[word] = negative_word_values.get(word, 0) + weight
        overall_word_values[word] = overall_word_values.get(word, 0) + weight

# Prepare data for output with an additional column for the formula
all_words = set(positive_word_values.keys()).union(set(negative_word_values.keys()))
data = []
for word in all_words:
    positive_weight = positive_word_values.get(word, 0)
    negative_weight = negative_word_values.get(word, 0)
    overall_weight = overall_word_values.get(word, 0)

    # Calculate the new column value using the provided formula
    new_column_value = np.log((positive_weight + 0.001) / (negative_weight + 0.001))

    data.append({
        "Term": word,
        "Attention Weight (Positive Review)": positive_weight,
        "Attention Weight (Negative Review)": negative_weight,
        "Attention Weight (Overall)": overall_weight,
        "Log Ratio": new_column_value  # New column added
    })

# Add predicted sentiment column to DataFrame
df["Predicted Sentiment"] = predicted_sentiments

# Calculate accuracy
df["Correct Prediction"] = df.apply(lambda row: 1 if row["Predicted Sentiment"] == row["sentiment"] else 0, axis=1)
accuracy = (df["Correct Prediction"].sum() / len(df)) * 100

# Calculate Precision, Recall, F1 Score
y_true = [1 if s == "positive" else 0 for s in actual_sentiments]
y_pred = [1 if s == "positive" else 0 for s in predicted_sentiments]

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

# Save predicted sentiments and accuracy results to CSV
df.to_csv("bert_predicted_sentiments.csv", index=False)

# Save metrics to a new CSV file
metrics_data = {
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Value": [accuracy, precision, recall, f1]
}
metrics_df = pd.DataFrame(metrics_data)
metrics_df.to_csv("bert_metrics.csv", index=False)

print(f"Model accuracy: {accuracy:.2f}%")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
df_output = pd.DataFrame(data)
df_output.to_csv("bert_attention_weights_amazon.csv", index=False)
print("CSV files 'bert_predicted_sentiments_amazon.csv' and 'bert_metrics_amazon.csv' updated successfully!")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

DistilBertSdpaAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


Error: The file 'sentiment_reviews.csv' was not found.
Model accuracy: 68.80%
Precision: 0.7516
Recall: 0.5105
F1 Score: 0.6080
CSV files 'bert_predicted_sentiments_amazon.csv' and 'bert_metrics_amazon.csv' updated successfully!


In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import RobertaTokenizer, RobertaForSequenceClassification
import nltk
from nltk.corpus import stopwords
from sklearn.metrics import precision_score, recall_score, f1_score

# Download stopwords if needed
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Load RoBERTa sentiment model
model_name = "cardiffnlp/twitter-roberta-base-sentiment"  # Pretrained sentiment model
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained(model_name, output_attentions=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Load dataset
try:
    df = pd.read_csv("yelp_reviews_sentiment.csv")
except FileNotFoundError:
    print("Error: The file 'sentiment_reviews.csv' was not found.")
    exit()
except pd.errors.EmptyDataError:
    print("Error: The file is empty.")
    exit()
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    exit()

# Function to extract attention weights
def get_attention_weights(attentions):
    last_layers = attentions[-2:]  # Take last 2 layers for better accuracy
    attn_scores = torch.mean(torch.stack(last_layers), dim=[0, 1, 2])  # Mean over layers & heads
    term_weights = attn_scores[0, 1:-1].cpu().numpy()  # Remove CLS & SEP
    term_weights /= term_weights.sum()  # Normalize
    return term_weights

# Function to process a review and extract word-level attention weights
def process_review(review):
    inputs = tokenizer(review, return_tensors="pt", truncation=True, max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        attentions = outputs.attentions

    attention_weights = get_attention_weights(attentions)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])[1:-1]  # Remove CLS & SEP

    word_values = {tok: attn for tok, attn in zip(tokens, attention_weights) if tok not in stop_words}
    return word_values

# Process positive and negative reviews
positive_reviews = df[df["sentiment"] == "positive"]["review"].tolist()
negative_reviews = df[df["sentiment"] == "negative"]["review"].tolist()

overall_word_values = {}
positive_word_values = {}
negative_word_values = {}

# Store predicted and actual sentiments
predicted_sentiments = []
actual_sentiments = df["sentiment"].tolist()

# Process all reviews and aggregate attention weights
for index, row in df.iterrows():
    review = row["review"]
    word_weights = process_review(review)
    log_weighted_score = sum(
        np.log((positive_word_values.get(word, 0) + 0.001) / (negative_word_values.get(word, 0) + 0.001))
        for word in word_weights.keys()
    )
    predicted_sentiment = "positive" if log_weighted_score > 0 else "negative"
    predicted_sentiments.append(predicted_sentiment)

    for word, weight in word_weights.items():
        if row["sentiment"] == "positive":
            positive_word_values[word] = positive_word_values.get(word, 0) + weight
        else:
            negative_word_values[word] = negative_word_values.get(word, 0) + weight
        overall_word_values[word] = overall_word_values.get(word, 0) + weight

# Prepare data for output with an additional column for the formula
all_words = set(positive_word_values.keys()).union(set(negative_word_values.keys()))
data = []
for word in all_words:
    positive_weight = positive_word_values.get(word, 0)
    negative_weight = negative_word_values.get(word, 0)
    overall_weight = overall_word_values.get(word, 0)

    # Calculate the new column value using the provided formula
    new_column_value = np.log((positive_weight + 0.001) / (negative_weight + 0.001))

    data.append({
        "Term": word,
        "Attention Weight (Positive Review)": positive_weight,
        "Attention Weight (Negative Review)": negative_weight,
        "Attention Weight (Overall)": overall_weight,
        "Log Ratio": new_column_value  # New column added
    })

# Add predicted sentiment column to DataFrame
df["Predicted Sentiment"] = predicted_sentiments

# Calculate accuracy
df["Correct Prediction"] = df.apply(lambda row: 1 if row["Predicted Sentiment"] == row["sentiment"] else 0, axis=1)
accuracy = (df["Correct Prediction"].sum() / len(df)) * 100

# Calculate Precision, Recall, F1 Score
y_true = [1 if s == "positive" else 0 for s in actual_sentiments]
y_pred = [1 if s == "positive" else 0 for s in predicted_sentiments]

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

# Save predicted sentiments and accuracy results to CSV
df.to_csv("yelp_roberta_predicted_sentiments.csv", index=False)

# Save metrics to a new CSV file
metrics_data = {
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Value": [accuracy, precision, recall, f1]
}
metrics_df = pd.DataFrame(metrics_data)
metrics_df.to_csv("yelp_roberta_metrics.csv", index=False)

print(f"Model accuracy: {accuracy:.2f}%")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print("CSV files 'yelp_roberta_predicted_sentiments.csv' and 'yelp_roberta_metrics.csv' updated successfully!")

df_output = pd.DataFrame(data)
df_output.to_csv("yelp_roberta_attention_weights.csv", index=False)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
RobertaSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be re

Model accuracy: 71.70%
Precision: 0.6861
Recall: 0.8000
F1 Score: 0.7387
CSV files 'yelp_roberta_predicted_sentiments.csv' and 'yelp_roberta_metrics.csv' updated successfully!


In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import RobertaTokenizer, RobertaForSequenceClassification
import nltk
from nltk.corpus import stopwords
from sklearn.metrics import precision_score, recall_score, f1_score

# Download stopwords if needed
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Load RoBERTa sentiment model
model_name = "cardiffnlp/twitter-roberta-base-sentiment"  # Pretrained sentiment model
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained(model_name, output_attentions=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Load dataset
try:
    df = pd.read_csv("amazon_reviews_sentiment.csv")
except FileNotFoundError:
    print("Error: The file 'sentiment_reviews.csv' was not found.")
    exit()
except pd.errors.EmptyDataError:
    print("Error: The file is empty.")
    exit()
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    exit()

# Function to extract attention weights
def get_attention_weights(attentions):
    last_layers = attentions[-2:]  # Take last 2 layers for better accuracy
    attn_scores = torch.mean(torch.stack(last_layers), dim=[0, 1, 2])  # Mean over layers & heads
    term_weights = attn_scores[0, 1:-1].cpu().numpy()  # Remove CLS & SEP
    term_weights /= term_weights.sum()  # Normalize
    return term_weights

# Function to process a review and extract word-level attention weights
def process_review(review):
    inputs = tokenizer(review, return_tensors="pt", truncation=True, max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        attentions = outputs.attentions

    attention_weights = get_attention_weights(attentions)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])[1:-1]  # Remove CLS & SEP

    word_values = {tok: attn for tok, attn in zip(tokens, attention_weights) if tok not in stop_words}
    return word_values

# Process positive and negative reviews
positive_reviews = df[df["sentiment"] == "positive"]["review"].tolist()
negative_reviews = df[df["sentiment"] == "negative"]["review"].tolist()

overall_word_values = {}
positive_word_values = {}
negative_word_values = {}

# Store predicted and actual sentiments
predicted_sentiments = []
actual_sentiments = df["sentiment"].tolist()

# Process all reviews and aggregate attention weights
for index, row in df.iterrows():
    review = row["review"]
    word_weights = process_review(review)
    log_weighted_score = sum(
        np.log((positive_word_values.get(word, 0) + 0.001) / (negative_word_values.get(word, 0) + 0.001))
        for word in word_weights.keys()
    )
    predicted_sentiment = "positive" if log_weighted_score > 0 else "negative"
    predicted_sentiments.append(predicted_sentiment)

    for word, weight in word_weights.items():
        if row["sentiment"] == "positive":
            positive_word_values[word] = positive_word_values.get(word, 0) + weight
        else:
            negative_word_values[word] = negative_word_values.get(word, 0) + weight
        overall_word_values[word] = overall_word_values.get(word, 0) + weight

# Prepare data for output with an additional column for the formula
all_words = set(positive_word_values.keys()).union(set(negative_word_values.keys()))
data = []
for word in all_words:
    positive_weight = positive_word_values.get(word, 0)
    negative_weight = negative_word_values.get(word, 0)
    overall_weight = overall_word_values.get(word, 0)

    # Calculate the new column value using the provided formula
    new_column_value = np.log((positive_weight + 0.001) / (negative_weight + 0.001))

    data.append({
        "Term": word,
        "Attention Weight (Positive Review)": positive_weight,
        "Attention Weight (Negative Review)": negative_weight,
        "Attention Weight (Overall)": overall_weight,
        "Log Ratio": new_column_value  # New column added
    })

# Add predicted sentiment column to DataFrame
df["Predicted Sentiment"] = predicted_sentiments

# Calculate accuracy
df["Correct Prediction"] = df.apply(lambda row: 1 if row["Predicted Sentiment"] == row["sentiment"] else 0, axis=1)
accuracy = (df["Correct Prediction"].sum() / len(df)) * 100

# Calculate Precision, Recall, F1 Score
y_true = [1 if s == "positive" else 0 for s in actual_sentiments]
y_pred = [1 if s == "positive" else 0 for s in predicted_sentiments]

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

# Save predicted sentiments and accuracy results to CSV
df.to_csv("amazon_roberta_predicted_sentiments.csv", index=False)

# Save metrics to a new CSV file
metrics_data = {
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Value": [accuracy, precision, recall, f1]
}
metrics_df = pd.DataFrame(metrics_data)
metrics_df.to_csv("amazon_roberta_metrics.csv", index=False)

print(f"Model accuracy: {accuracy:.2f}%")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print("CSV files 'amazon_roberta_predicted_sentiments.csv' and 'amazon_roberta_metrics.csv' updated successfully!")

df_output = pd.DataFrame(data)
df_output.to_csv("amazon_roberta_attention_weights.csv", index=False)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

RobertaSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Model accuracy: 72.70%
Precision: 0.7239
Recall: 0.7340
F1 Score: 0.7289
CSV files 'amazon_roberta_predicted_sentiments.csv' and 'amazon_roberta_metrics.csv' updated successfully!
